In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

## Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
train = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
test = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")

In [ ]:
print("Train Shape:", train.shape)
print("Test Shape:", test.shape)

In [ ]:
!pip install sentence-transformers -q

In [ ]:
import torch
import torch.nn as nn

In [ ]:
from sentence_transformers import SentenceTransformer

In [ ]:
class MLPClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(384,256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256,128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128,64),
            nn.ReLU(),
            nn.Linear(64,1)
        )
    def forward(self,x):
        return self.network(x)

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model = MLPClassifier().to(device)
model.load_state_dict(
    torch.load(
        "/kaggle/input/datasets/venkat23f1000054/mlp-minilm-model/mlp_model.pt",
        map_location=device
    )
)
model.eval()

In [ ]:
embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

In [ ]:
options = ["A","B","C","D","E"]
predictions = []
for _, row in test.iterrows():
    texts = [
        row["prompt"] + " [SEP] " + str(row[o])
        for o in options
    ]
    emb = embedder.encode(texts)
    X = torch.tensor(
        emb,
        dtype=torch.float32
    ).to(device)
    with torch.no_grad():
        probs = torch.sigmoid(
            model(X)
        ).squeeze().cpu().numpy()
    ranked = np.argsort(probs)[::-1]
    top3 = [options[i] for i in ranked[:3]]
    predictions.append(
        " ".join(top3)
    )

In [ ]:
sample = pd.read_csv(
    "/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv"
)
sample["Prediction"] = predictions
sample.head()

In [ ]:
sample.to_csv("submission.csv", index=False)